# Notebook 01 — Data Collection

This notebook assembles raw data from three sources:
1. **IRS 990-PF XML filings** (public S3 bucket — no auth required)
2. **ProPublica Nonprofit Explorer API** (no auth required)
3. **Candid Demographics + Premier APIs** (stub — activate after API key arrives)

All raw outputs are saved to `data/raw/` and are gitignored.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import time
import logging
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

from src.api_client import (
    fetch_990_index,
    download_990pf_xml,
    parse_990pf_grants,
    parse_990pf_funder_summary,
    propublica_organization,
    propublica_search,
)

logging.basicConfig(level=logging.INFO)
RAW = Path('../data/raw')
RAW.mkdir(exist_ok=True)

## 1  IRS 990-PF: Download Index Files

We pull the IRS e-file index for each year 2011–2023, which lists every electronically filed 990-PF with its S3 ObjectId.
We then download a sample of the actual XML filings to extract grant data.

> **Sampling strategy:** The full index for a single year can contain 50,000+ 990-PF filings. For this analysis we sample up to `MAX_FILINGS_PER_YEAR` per year, prioritizing foundations with the largest reported total assets (derived from the index metadata where available).

In [ ]:
YEARS = range(2011, 2024)   # IRS e-file data starts ~2011
MAX_FILINGS_PER_YEAR = 200  # Increase for a larger dataset; start small to test

all_index_rows = []

for year in YEARS:
    try:
        rows = fetch_990_index(year)
        # Attach year tag and store
        for r in rows:
            r['index_year'] = year
        all_index_rows.extend(rows)
        print(f"{year}: {len(rows):,} 990-PF filings")
    except Exception as e:
        print(f"{year}: ERROR — {e}")
    time.sleep(0.5)  # polite delay

index_df = pd.DataFrame(all_index_rows)
index_df.to_csv(RAW / '990pf_index.csv', index=False)
print(f"\nTotal index rows saved: {len(index_df):,}")
index_df.head()

In [ ]:
# Sample filings to download
sample = (
    index_df
    .groupby('index_year', group_keys=False)
    .apply(lambda g: g.sample(min(MAX_FILINGS_PER_YEAR, len(g)), random_state=42))
)
print(f"Sampling {len(sample):,} filings across {sample['index_year'].nunique()} years")

In [ ]:
grants_records = []
funder_records = []
errors = []

for _, row in tqdm(sample.iterrows(), total=len(sample), desc='Downloading 990-PFs'):
    obj_id = row.get('ObjectId')
    if not obj_id:
        continue
    try:
        xml_text = download_990pf_xml(obj_id, save=True)
        grants = parse_990pf_grants(xml_text)
        for g in grants:
            g['object_id'] = obj_id
        grants_records.extend(grants)
        funder_records.append(parse_990pf_funder_summary(xml_text))
        time.sleep(0.1)  # polite delay
    except Exception as e:
        errors.append({'object_id': obj_id, 'error': str(e)})

print(f"Downloaded: {len(funder_records):,} filings, {len(grants_records):,} grant records, {len(errors)} errors")

In [ ]:
grants_raw = pd.DataFrame(grants_records)
funders_raw = pd.DataFrame(funder_records)

grants_raw.to_csv(RAW / 'grants_raw.csv', index=False)
funders_raw.to_csv(RAW / 'funders_raw.csv', index=False)
pd.DataFrame(errors).to_csv(RAW / 'download_errors.csv', index=False)

print(f"Grants raw shape: {grants_raw.shape}")
grants_raw.head()

## 2  ProPublica: Enrich Recipient Data

For each unique recipient EIN in our grants data, we query ProPublica to get NTEE code, revenue, and location details.

In [ ]:
recipient_eins = grants_raw['recipient_ein'].dropna().unique()
print(f"Unique recipient EINs to enrich: {len(recipient_eins):,}")

recipient_records = []

for ein in tqdm(recipient_eins[:500], desc='ProPublica lookups'):  # cap for initial run
    try:
        data = propublica_organization(ein)
        org = data.get('organization', {})
        recipient_records.append({
            'ein': ein,
            'name': org.get('name'),
            'ntee_code': org.get('ntee_code'),
            'state': org.get('state'),
            'city': org.get('city'),
            'total_revenue': org.get('revenue_amount'),
            'subsection_code': org.get('subsection_code'),
        })
        time.sleep(0.3)  # ProPublica rate limit is generous but be polite
    except Exception as e:
        recipient_records.append({'ein': ein, 'error': str(e)})

recipients_raw = pd.DataFrame(recipient_records)
recipients_raw.to_csv(RAW / 'recipients_raw.csv', index=False)
print(f"Recipients enriched: {len(recipients_raw):,}")
recipients_raw.head()

## 3  Candid APIs (Stub)

These cells will activate once the Candid API key is available.
Set the environment variable `CANDID_API_KEY` in a `.env` file at the project root.

In [ ]:
# Uncomment and run after setting CANDID_API_KEY

# from dotenv import load_dotenv
# load_dotenv('../.env')
#
# from src.api_client import candid_demographics, candid_grants_search
#
# demo_records = []
# for ein in tqdm(recipient_eins[:200], desc='Candid demographics'):
#     try:
#         data = candid_demographics(ein)
#         demo_records.append(data)
#         time.sleep(0.5)
#     except Exception as e:
#         print(f"  {ein}: {e}")
#
# pd.DataFrame(demo_records).to_csv(RAW / 'demographics_raw.csv', index=False)
print('Candid API stub — set CANDID_API_KEY to activate')

## Summary

Raw data saved to `data/raw/`:
- `990pf_index.csv` — IRS index of all 990-PF filings 2011–2023
- `990pf/` — individual XML filings
- `grants_raw.csv` — extracted grant records
- `funders_raw.csv` — funder summaries
- `recipients_raw.csv` — ProPublica-enriched recipient data

Proceed to **Notebook 02** for cleaning and loading into SQLite.